<img src="https://github.com/nicholasmetherall/digital-earth-pacific-macblue-activities/blob/main/attachments/images/DE_Pacific_banner.JPG?raw=true" width="900"/>

Figure 1.1.a. Jupyter environment + Python notebooks

# Digital Earth Pacific Notebook 1B prepare postcard and load data to csv

The objective of this notebook is to prepare a geomad postcard for your AOI (masking, scaling and loading additional band ratios and spectral indices) and sampling all the datasets into a csv based on your training data geodataframe.

## Step 1.1: Configure the environment

In [1]:
import os
from datetime import datetime
from shapely.geometry import Polygon
from shapely import box
from pyproj import CRS 
import folium
from odc.geo import Geometry
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio as rio
import xarray as xr
import rioxarray
from ipyleaflet import basemaps
from numpy.lib.stride_tricks import sliding_window_view
import pystac_client
from dask.distributed import Client as DaskClient
from odc.stac import load, configure_s3_access
import planetary_computer
# from odc.stac import load
from pystac.client import Client
from skimage.feature import graycomatrix, graycoprops
from utils import load_data, load_s1_dem, scale, calculate_band_indices, apply_mask, mask_water, all_masks, do_prediction

In [2]:
# Reload scripts and imports
%load_ext autoreload
%autoreload 2

<font color='red'>1.1. Revise the site of training data to 'usp'


In [3]:
site = "usp"
filename = f"training-data/{site}-processed-tdata.geojson"

## Step 1.2: Configure STAC access and search parameters

In [4]:
catalog = "https://stac.digitalearthpacific.org"
client = Client.open(catalog)

<font color='red'>1.2. Drag and drop the Revise the site of the area of interest to 'usp'

In [5]:
area_of_interest = "usp"

# 1. Read the file
aoi = gpd.read_file(f"{filename}")

# 2. Get the bounding box directly from the GeoDataFrame
# This returns exactly what STAC needs: (minx, miny, maxx, maxy)
bbox = aoi.total_bounds 

<font color='red'>1.3. Revise the year to '2025'

In [6]:
# 3. Proceed with your search
year = '2025'
items = list(
    client.search(
        collections=["dep_s2_geomad"],
        bbox=bbox,
        datetime=year
    ).items()
)

#### How many images were found?

In [7]:
print(f"Found {len(items)} items in for {datetime}")

Found 1 items in for <class 'datetime.datetime'>


<font color='red'>1.4. Define the spectral bands you are interested in as:
> `measurements = measurements = ["nir", "red", "blue", "green", "emad", "smad", "bcmad", "green", "nir08", "nir09", "swir16", "swir22", "coastal", "rededge1", "rededge2", "rededge3"]`

In [8]:
measurements = measurements = ["nir", "red", "blue", "green", "emad", "smad", "bcmad", "green", "nir08", "nir09", "swir16", "swir22", "coastal", "rededge1", "rededge2", "rededge3"]

<font color='red'>1.5. Load the data

In [9]:
data = load_data(
    items,
    measurements,
    bbox,
)

In [10]:
# dask_client = DaskClient(n_workers=1, threads_per_worker=16, memory_limit='32GB')
# configure_s3_access(cloud_defaults=True, requester_pays=True)

<font color='red'>1.6. Scale the variable 'data'
>scaled = scale(`insertvariable`)

In [11]:
scaled = scale(data)

In [12]:
scaled = scaled.squeeze()

<font color='red'>1.7. Explore the new 'scaled' variable 
>`insertvariable`.odc.explore(vmin=0, vmax=0.3, bands=["red", "green", "blue"], crs="EPSG:3832", name=site)

In [13]:
scaled.odc.explore(vmin=0, vmax=0.3, bands=["red", "green", "blue"], crs="EPSG:3832", name=site)

/srv/conda/envs/notebook/lib/python3.11/site-packages/rasterio/warp.py:387: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dest = _reproject(
/srv/conda/envs/notebook/lib/python3.11/site-packages/rasterio/warp.py:387: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dest = _reproject(


<font color='red'>1.8. Calculate the spectral bands for this 'scaled' variable 
>`variable` = calculate_band_indices(`variable`)  

In [14]:
scaled = calculate_band_indices(scaled)

In [15]:
# see results
scaled

<xarray.Dataset> Size: 2MB
Dimensions:        (y: 90, x: 77)
Coordinates:
  * y              (y) float64 720B -2.042e+06 -2.042e+06 ... -2.043e+06
  * x              (x) float64 616B 3.167e+06 3.167e+06 ... 3.168e+06 3.168e+06
    spatial_ref    int32 4B 3832
    time           datetime64[ns] 8B 2025-01-01
Data variables: (12/32)
    nir            (y, x) float64 55kB dask.array<chunksize=(90, 77), meta=np.ndarray>
    red            (y, x) float64 55kB dask.array<chunksize=(90, 77), meta=np.ndarray>
    blue           (y, x) float64 55kB dask.array<chunksize=(90, 77), meta=np.ndarray>
    green          (y, x) float64 55kB dask.array<chunksize=(90, 77), meta=np.ndarray>
    emad           (y, x) float32 28kB dask.array<chunksize=(90, 77), meta=np.ndarray>
    smad           (y, x) float32 28kB dask.array<chunksize=(90, 77), meta=np.ndarray>
    ...             ...
    ndci           (y, x) float64 55kB dask.array<chunksize=(90, 77), meta=np.ndarray>
    nbi            (y, x) float64 55kB dask.array<chunksize=(90, 77), meta=np.ndarray>
    ndmi           (y, x) float64 55kB dask.array<chunksize=(90, 77), meta=np.ndarray>
    bsi            (y, x) float64 55kB dask.array<chunksize=(90, 77), meta=np.ndarray>
    awei           (y, x) float64 55kB dask.array<chunksize=(90, 77), meta=np.ndarray>
    tc_wetness     (y, x) float64 55kB dask.array<chunksize=(90, 77), meta=np.ndarray>

<font color='red'>1.9. Mask out the 'scaled' variable including the ocean 
>`scaled`, mask = all_masks(`scaled`, return_mask = True)


In [16]:
scaled, mask = all_masks(scaled, return_mask = True)


<font color='red'>Explore the extent of this 'mask' variable  
>`insertvariable`.odc.explore(vmin=0, vmax=0.3, bands=["red", "green", "blue"], crs="EPSG:3832", name=site)

In [17]:
scaled.odc.explore(vmin=0, vmax=0.3, bands=["red", "green", "blue"], crs="EPSG:3832", name=site)

### Postcard csv

The objective of this notebook was to train the machine learning model that will allow us to classify an area with land cover classes defined through the training data.

Step 1.2. Input the training data to sample geomad data from the postcard.

`training = gpd.read_file(f"training-data/usp-processed-tdata.geojson")`

In [18]:
training = gpd.read_file(f"training-data/usp-processed-tdata.geojson")

<font color='red'>2.1. Explore the new clipped extent of the 'training' data within the extent of your aoi  
> `variable`.explore()

In [19]:
training.explore()

In [20]:
# Reproject training data to the GeoMAD CRS and convert to xarray
training_reprojected = training.to_crs(scaled.odc.crs)
training_da = training_reprojected.assign(
    x=training_reprojected.geometry.x, y=training_reprojected.geometry.y
).to_xarray()

# Extract training values from the masked dataset
training_values = (
    scaled.sel(training_da[["x", "y"]], method="nearest")
    .squeeze()
    .compute()
    .to_pandas()
)
# training_values

<font color='red'>2.2. You have made a new variable called 'training values'... To better understand what is inside this inspect the columns  
> `variable`.columns

In [21]:
training_values.columns

Index(['nir', 'red', 'blue', 'green', 'emad', 'smad', 'bcmad', 'nir08',
       'nir09', 'swir16', 'swir22', 'coastal', 'rededge1', 'rededge2',
       'rededge3', 'mndwi', 'ndti', 'cai', 'ndvi', 'evi', 'savi', 'ndwi',
       'b_g', 'b_r', 'swir22_swir16', 'mci', 'ndci', 'nbi', 'ndmi', 'bsi',
       'awei', 'tc_wetness', 'y', 'x', 'spatial_ref', 'time'],
      dtype='object')

<font color='red'>2.3. Run the following lines of code unchanged to generate more information about the data array you are using to train the machine learning model  

In [22]:
# Join the training data with the extracted values and remove unnecessary columns
training_array = pd.concat([training["LULC_code"], training_values], axis=1)

# Drop rows where there was no data available
training_array = training_array.dropna()

# Preview our resulting training array
# training_array.head()

In [23]:
print(training_array.shape[1], 'total columns')
# print('columns included', training_array.columns)

37 total columns


In [24]:
print(training_array['LULC_code'].value_counts())
print('total gps points',(len(training_array)))

LULC_code
5.0    134
3.0    115
2.0     71
6.0     41
Name: count, dtype: int64
total gps points 361


In [25]:
training_array=training_array.drop(columns=["spatial_ref", "time"])

<font color='red'>2.4. You have made a final `training_array` variable. Inspect it here by entering it below:

In [26]:
training_array

,LULC_code,nir,red,blue,green,emad,smad,bcmad,nir08,nir09,...,swir22_swir16,mci,ndci,nbi,ndmi,bsi,awei,tc_wetness,y,x
0,5.0,0.2507,0.1590,0.1276,0.1571,0.077371,8.668303e-08,0.000005,0.3251,0.2923,...,0.697758,1.483432,0.030488,-0.058476,0.058476,-0.091932,-0.151300,-0.001695,-2041715.0,3167305.0
1,3.0,0.4327,0.0560,0.0521,0.0911,0.090505,8.771121e-08,0.000005,0.3510,0.3126,...,0.652584,2.896252,0.454722,-0.320818,0.320818,-0.413411,-0.666650,-0.081518,-2041705.0,3167325.0
2,5.0,0.3480,0.0790,0.0655,0.0944,0.083219,1.416385e-07,0.000005,0.3727,0.3672,...,0.679682,1.959459,0.384256,-0.137627,0.137627,-0.231021,-0.571375,-0.088612,-2041735.0,3167335.0
3,3.0,0.3855,0.0645,0.0530,0.0848,0.076803,1.612633e-07,0.000005,0.3780,0.3658,...,0.644455,2.332123,0.438642,-0.275223,0.275223,-0.361378,-0.606600,-0.067343,-2041755.0,3167355.0
4,5.0,0.3617,0.0870,0.0674,0.0984,0.085797,2.139389e-07,0.000005,0.4145,0.3702,...,0.589357,2.559802,0.237845,-0.289713,0.289713,-0.354696,-0.498600,-0.046525,-2041765.0,3167365.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
441,3.0,0.4639,0.0806,0.0723,0.1215,0.139463,1.296997e-07,0.000006,0.5096,0.4527,...,0.525999,2.701805,0.361078,-0.242700,0.242700,-0.400914,-0.706675,-0.085801,-2041815.0,3167465.0
442,3.0,0.4734,0.0790,0.0651,0.1147,0.081405,1.371950e-07,0.000004,0.5169,0.4085,...,0.517107,2.582651,0.397636,-0.213379,0.213379,-0.387529,-0.778925,-0.099220,-2041805.0,3167515.0
443,6.0,0.1672,0.0819,0.0725,0.0929,0.094341,7.982015e-07,0.000012,0.1338,0.2623,...,0.645768,1.855716,0.047674,-0.271967,0.271967,-0.250391,-0.074150,0.022941,-2041825.0,3167565.0
444,6.0,0.1065,0.0792,0.0720,0.0887,0.088480,5.522847e-07,0.000008,0.1875,0.4008,...,0.625000,0.951743,0.171115,0.085444,-0.085444,-0.060291,-0.035850,0.013247,-2041835.0,3167545.0


<font color='red'>2.5. Export your final `training_array` variable to a CSV file.
>`variable`.to_csv(f"training-data/{site}-tdata.csv", index=False)

In [28]:
training_array.to_csv(f"training-data/{site}-tdata.csv", index=False)